Conduct ANOVA and post-hoc Tukey tests to compare means across multiple groups, interpreting F-statistics and p-values to validate assumptions of statistical significance in experimental data.

In [1]:
import pandas as pd
import numpy as np

In [2]:
np.random.seed(42)

In [3]:
sales_A = np.random.normal(loc=100, scale=10, size=40)
sales_B = np.random.normal(loc=115, scale=10, size=40)
sales_C = np.random.normal(loc=102, scale=10, size=40)

In [4]:
df_sales = pd.DataFrame({
    'sales': np.concatenate([sales_A, sales_B, sales_C]),
    'strategy': ['Strategy_A']*40 + ['Strategy_B']*40 + ['Strategy_C']*40
})

In [5]:
print("First few rows of dataset:")
print(df_sales.head())
print("\nMean sales per strategy:")
print(df_sales.groupby('strategy')['sales'].mean())

First few rows of dataset:
        sales    strategy
0  104.967142  Strategy_A
1   98.617357  Strategy_A
2  106.476885  Strategy_A
3  115.230299  Strategy_A
4   97.658466  Strategy_A

Mean sales per strategy:
strategy
Strategy_A     97.813632
Strategy_B    114.709565
Strategy_C    102.101731
Name: sales, dtype: float64


In [6]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [7]:
model = ols('sales ~ C(strategy)', data=df_sales).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

In [8]:
print("--- ANOVA Results ---")
print(anova_table)

--- ANOVA Results ---
                   sum_sq     df          F        PR(>F)
C(strategy)   6170.904718    2.0  35.851957  7.172815e-13
Residual     10069.127362  117.0        NaN           NaN


In [9]:
f_stat = anova_table.loc['C(strategy)', 'F']
p_val = anova_table.loc['C(strategy)', 'PR(>F)']

In [10]:
print(f"\nF-Statistic: {round(f_stat, 4)}")
print(f"P-Value: {round(p_val, 4)}")


F-Statistic: 35.852
P-Value: 0.0


In [11]:
if p_val < 0.05:
    print("Decision: Reject Null Hypothesis (H0). At least one strategy mean is significantly different!")
else:
    print("Decision: Fail to Reject Null Hypothesis (H0). All strategy means are similar.")

Decision: Reject Null Hypothesis (H0). At least one strategy mean is significantly different!


In [12]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [13]:
tukey_results = pairwise_tukeyhsd(
    endog=df_sales['sales'],
    groups=df_sales['strategy'],
    alpha=0.05
)

In [14]:
print("--- Post-Hoc Tukey HSD Results ---")
print(tukey_results)

--- Post-Hoc Tukey HSD Results ---
     Multiple Comparison of Means - Tukey HSD, FWER=0.05     
  group1     group2   meandiff p-adj   lower    upper  reject
-------------------------------------------------------------
Strategy_A Strategy_B  16.8959    0.0  11.9715 21.8203   True
Strategy_A Strategy_C   4.2881 0.1012  -0.6363  9.2125  False
Strategy_B Strategy_C -12.6078    0.0 -17.5322 -7.6834   True
-------------------------------------------------------------


In [15]:
tukey_df = pd.DataFrame(
    data=tukey_results._results_table.data[1:],
    columns=tukey_results._results_table.data[0]
)

In [16]:
print("Pairwise Comparison Summary:")
print(tukey_df[['group1', 'group2', 'meandiff', 'p-adj', 'reject']])

Pairwise Comparison Summary:
       group1      group2  meandiff   p-adj  reject
0  Strategy_A  Strategy_B   16.8959  0.0000    True
1  Strategy_A  Strategy_C    4.2881  0.1012   False
2  Strategy_B  Strategy_C  -12.6078  0.0000    True
